# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Beshair-Khan/flyrank_ml_internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

* **Unit Of Analysis:** 1 unique row=1 unique webpage identified by content_id
* **Time Windows:** We use march 2026 mid panel window
* **Table Used:** content_refresh_anonymized.csv
* **Predict / Rank:** opportunity_score (a 0–100 proxy ranking pages for content refresh priority).

In [41]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load dataset via Hugging Face repo path (or local fallback)
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
print(f"Loaded {len(df):,} pages from the warehouse release slice.")
total_rows=len(df)
unique_pages=df['content_id'].nunique()
print(f"Total rows are {total_rows:,}")
print(f"Unique Pages are {unique_pages:,}")
print(f"Grain Verified: {'PASSED(1 row = 1 unique page)' if total_rows==unique_pages else 'failed {dupicates found}'}")
df.head()

Loaded 30,000 pages from the warehouse release slice.
Total rows are 30,000
Unique Pages are 30,000
Grain Verified: PASSED(1 row = 1 unique page)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.

 **Features**
 * search_volume
 * cpc
 * avg_position
 * word_count
 * trend_pct
 * Features are knowable at the decision moment (March 2026) because search volume, rankings, word count, and trend slopes are logged historical metrics from Search Console/Analytics prior to making refresh decisions.

 **Label**
 * opportunity_score: Constructed proxy priority score

 **Context**
 * content_id: Unique anonymized webpage identifier.
 * main_intent: Categorical intent (informational, transactional, commercial).

 **Excluded**
 * url, client_name, domain: Excluded for privacy compliance (anonymization).
 * Future traffic/rankings: Excluded to avoid temporal data leakage (using future outcome states to predict current priority).

In [42]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['is_declining']=(df['trend_direction']=='down').astype(int)
df['is_page_one']=(df['position_tier']=='page_1').astype(int)
df['is_high_value']=(df['cpc']>df['cpc'].median()).astype(int)
df['opportunity_score'] = ((df['is_declining'] * 35) +(df['is_page_one'] * 25) +(df['is_high_value'] * 20) +((1 - df['ctr'].fillna(0)) * 20)).round(1)
features = ['search_volume', 'avg_position', 'cpc', 'word_count', 'trend_pct']
label = ['opportunity_score']
context = ['content_id', 'client_id', 'main_intent']
print("Field Buckets Assigned Successfully:")
print(f"Features ({len(features)}): {features}")
print(f"Label (1): {label}")
print(f"Context ({len(context)}): {context}")

Field Buckets Assigned Successfully:
Features (5): ['search_volume', 'avg_position', 'cpc', 'word_count', 'trend_pct']
Label (1): ['opportunity_score']
Context (3): ['content_id', 'client_id', 'main_intent']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*
* **Grain Check:** Confirming zero duplicate content_id entries.
* **Row Count & Completeness:** Measuring overall row count and non-null availability across features.
* **Availability Filter (IS TRUE check):** Verifying how many rows survive when requiring valid, non-null feature values.

In [43]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
dup=df.duplicated(subset=['content_id']).sum()
print(f"The duplicate grains are {dup}")
print(f"\nThe total rows are {len(df):,}")
print("\nChecking null values in features")
print(df[features].notnull().sum())
is_complete_row = df[features].notnull().all(axis=1) & df['content_id'].notnull()
surviving_rows = is_complete_row.sum()
survival_pct = (surviving_rows / total_rows) * 100
print("\nAVAILABILITY FILTER (IS TRUE)")
print(f"Feature columns checked: {features}")
print(f"Surviving rows after filter: {surviving_rows:,} / {total_rows:,} ({survival_pct:.1f}% survived)")

The duplicate grains are 0

The total rows are 30,000

Checking null values in features
search_volume    27532
avg_position     30000
cpc              27532
word_count       22301
trend_pct        26612
dtype: int64

AVAILABILITY FILTER (IS TRUE)
Feature columns checked: ['search_volume', 'avg_position', 'cpc', 'word_count', 'trend_pct']
Surviving rows after filter: 18,013 / 30,000 (60.0% survived)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**What can this data never tell you?**

This dataset cannot observe external search engine algorithm updates or competitor backlink changes. A drop in traffic might be driven by external core updates rather than content staleness.

In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df['cheat'] = df['opportunity_score'] * 0.99
print("Cheat Correlation:", df['cheat'].corr(df['opportunity_score']))
df.drop(columns=['cheat'], inplace=True)
print("cheat column is removed")

Cheat Correlation: 1.0
cheat column is removed


## Self-check

Before you submit, confirm each line honestly:

- [ Yes ] Every section above is filled — markdown thinking AND the code that backs it
- [ Yes ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ Yes ] No client names, URLs, or private queries anywhere
- [ Yes ] My claims use careful words: observed, measured, directional, decision-support
- [ Yes ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.